# PE zero-shot classification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/finished/colabs/04_pe_zero_shot.ipynb)

After the SigLIP/DINO ensemble, I tried **PE-Core-G14-448** as a single frozen image-text model. I used the same 80 public text templates per class and averaged full-image and horizontal-flip scores.

`submission_pe_zero_shot.csv` reached **0.96776** on the public leaderboard, my best result in this series.

This notebook keeps the preprocessing checks, saved-result analysis and inference in one place. `RUN_INFERENCE=False` reads an existing run from Drive; set it to `True` to generate predictions in a new run directory.


## 1. Domain shift

The training and validation images are 32×32 CIFAR-10 images. The synthetic test set includes different representations of the same categories, such as drawings and objects made from different materials. Strong clean validation performance did not consistently transfer to this test set in my earlier runs.

I use the competition files and public pretrained weights and templates. CIFAR-10-Warehouse is excluded.


## 2. Results leading to PE

| Experiment | Public Kaggle accuracy |
|---|---:|
| ResNet-18 baseline | 61.093% |
| ResNet-18 + AugMix | 75.796% |
| DINOv3 ConvNeXt linear probe | 82.269% |
| DINOv3 ConvNeXt partial fine-tuning | 86.217% |
| SigLIP 2 Giant probe | 92.019% |
| DINOv3 ViT-7B probe | 92.978% |
| SigLIP 2 Giant zero-shot | 94.015% |
| SigLIP/DINO ensemble | 94.844% |
| **PE zero-shot + flip** | **96.776%** |

PE improved on the ensemble by **1.932 percentage points**, reducing its remaining errors by about **37.5%**. I kept PE as the best-scoring model. These comparisons change pretraining, architecture and preprocessing together, so I cannot assign the gain to any single factor.


## 3. Architecture

PE maps images and text into a shared embedding space. I reuse both pretrained encoders without fine-tuning.

| Component | PE-Core-G14-448 |
|---|---|
| Visual encoder | ViT, approximately 1.88B parameters |
| Input | 448×448 RGB |
| Patch size | 14×14, giving 1,024 patch tokens |
| Visual transformer | 50 blocks, width 1,536, 16 attention heads, MLP width 8,960 |
| Position encoding | Absolute embeddings and 2D RoPE |
| Visual pooling | 8-head attention pooling, no CLS token in this variant |
| Text encoder | Approximately 0.47B parameters, 24 blocks, width 1,280, 20 heads |
| Text context | Up to 72 tokens |
| Shared embedding | 1,280 dimensions |

The dimensions come from the [pinned model configuration](https://github.com/facebookresearch/perception_models/blob/3e352cca660658d4b5c90f42a7808b11469e4c66/core/vision_encoder/config.py). Resizing creates a larger patch grid, but the original image still contains only 32×32 pixels of information.


In [ ]:
from IPython.display import display, HTML
# Sketch of the image and text branches.
def architecture_svg():
    boxes = [
      (20,25,'32 × 32 image','Bilinear → 448 × 448'),
      (20,125,'1,024 patches of 14 × 14','Projection to width 1,536 + position'),
      (20,225,'50 visual transformer blocks','Attention + MLP + residual connections'),
      (20,325,'Attention pooling + projection','Normalized visual vector: 1,280'),
      (410,25,'10 classes × 80 public templates','Tokenizer: context length 72'),
      (410,125,'24 text transformer blocks','Vector per text: 1,280'),
      (410,225,'Normalize → average → normalize','10 prototypes of dimension 1,280'),
      (410,325,'10 cosine similarities per view','Average original/flip → argmax')]
    svg = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 750 415" role="img" aria-label="PE architecture" style="max-width:100%;background:#f8fafc"><defs><marker id="arr" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#475569"/></marker></defs>'
    for x,y,title,subtitle in boxes:
        svg += f'<rect x="{x}" y="{y}" width="310" height="65" rx="10" fill="white" stroke="#2563eb"/><text x="{x+155}" y="{y+25}" text-anchor="middle" font-family="sans-serif" font-size="13" font-weight="bold">{title}</text><text x="{x+155}" y="{y+48}" text-anchor="middle" font-family="sans-serif" font-size="11">{subtitle}</text>'
    for x in (175,565):
        for y in (90,190,290):
            svg += f'<path d="M{x},{y} L{x},{y+30}" stroke="#475569" stroke-width="2" marker-end="url(#arr)"/>'
    return svg + '<path d="M330,357 L403,357" stroke="#475569" stroke-width="2" marker-end="url(#arr)"/></svg>'
display(HTML(architecture_svg()))

## 4. Colab setup

Saved predictions and image plots can be inspected without loading PE. Inference needs a BF16-capable GPU, preferably an A100. I keep the original results directory separate from a new inference run.

The setup keeps Colab's PyTorch/torchvision pair and checks the hashes of the experiment modules.


In [ ]:
%pip -q install "transformers==4.57.3" "huggingface_hub>=0.34,<1" "timm==1.0.15" "einops==0.8.1" "ftfy==6.3.1" regex kagglehub
%pip -q install matplotlib pandas pillow

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, sys
import requests

SOURCES = {
    'semantic_vit': ('fc538b90a48c2c1ce7cd8f3f96045653478744e9', '8d3baa53881282e34cbdb21c3527c7164213f3e78c1d921f393193c011958afc'),
    'pe_semantic': ('c30fc5422f23728a42cdc18741dac7ef52a7154c', '00f6ede2beb99c5abe82e3a6813e06f1802d2dedca3b82576fb2477ce951c214'),
}
CODE_DIR = Path('/content/recvis_pe_code')
CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, (revision, expected_hash) in SOURCES.items():
    url = f'https://raw.githubusercontent.com/rivas-vega/A3_Recv/{revision}/{name}.py'
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    if hashlib.sha256(response.content).hexdigest() != expected_hash:
        raise RuntimeError(f'File {name} does not match the pinned revision.')
    path = CODE_DIR/f'{name}.py'
    path.write_bytes(response.content)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
import semantic_vit as sv
import pe_semantic as pe
print('Verified code revision:', SOURCES)

In [ ]:
from google.colab import drive
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/A3_Recv')
PREVIOUS_PE_RUN = 'pe_g14_448_v1'
PREVIOUS_DIR = BASE/'runs'/PREVIOUS_PE_RUN
NEW_RUN_DIR = BASE/'runs'/'pe_zero_shot_didactic_v1'
RUN_INFERENCE = False  # True: run inference; False: inspect saved predictions.
SAMPLE_SEED = 42
print('Saved run:', PREVIOUS_DIR)
print('New inference run:', NEW_RUN_DIR)
print('GPU:', torch.cuda.get_device_name() if torch.cuda.is_available() else 'No GPU; saved-result analysis is available')

## 5. Load the competition data

Use the `KAGGLE_API_TOKEN` Colab secret or KaggleHub login. The loader checks array shapes and IDs. `SAMPLE_SEED` controls the small random samples used in the plots.


In [ ]:
import kagglehub
from google.colab import userdata
if not os.environ.get('KAGGLE_API_TOKEN'):
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
    del token
kagglehub.login()
DATA_ROOT = Path(kagglehub.competition_download('mva-recvis-2026'))
data = sv.read_data(DATA_ROOT)
CLASSES = sv.CLASSES
for split in ('train','val','test'):
    print(split, data[split].shape, data[split].dtype)

## 6. A small train/test sample

Eight random images from each split. Training panels include the provided labels; test panels show IDs only. Matplotlib's nearest-neighbor display keeps the original pixels visible.


In [ ]:
rng = np.random.default_rng(SAMPLE_SEED)
train_idx = rng.choice(len(data['train']), size=min(8,len(data['train'])), replace=False)
test_idx = rng.choice(len(data['test']), size=min(8,len(data['test'])), replace=False)
fig,axes = plt.subplots(2,8,figsize=(16,4.5),squeeze=False)
for row,(split,indices) in enumerate((('train',train_idx),('test',test_idx))):
    for ax in axes[row]: ax.axis('off')
    for ax,i in zip(axes[row],indices):
        ax.imshow(data[split][i],interpolation='nearest')
        title = f"train: {CLASSES[int(data['train_labels'][i])]}" if split=='train' else f"test ID {int(data['test_ids'][i])}"
        ax.set_title(title,fontsize=10)
fig.suptitle('Random train and test samples')
plt.tight_layout(); plt.show()

## 7. Bilinear resizing

PE receives a direct **32 → 448** bilinear resize. The 64×64 and 128×128 panels are visual comparisons only, not intermediate steps in the inference pipeline.

All panels use nearest-neighbor display after PIL resizing so that Matplotlib does not add another smoothing step.


In [ ]:
def bilinear_rgb(array,size):
    return Image.fromarray(np.asarray(array,dtype=np.uint8)).convert('RGB').resize((size,size),Image.Resampling.BILINEAR)

fig,axes = plt.subplots(4,5,figsize=(15,11),squeeze=False)
for ax in axes.flat: ax.axis('off')
for row,i in enumerate(test_idx[:4]):
    original = Image.fromarray(data['test'][i]).convert('RGB')
    target = bilinear_rgb(data['test'][i],448)
    views = [original,bilinear_rgb(data['test'][i],64),bilinear_rgb(data['test'][i],128),target,ImageOps.mirror(target)]
    titles = ['Original 32×32','Bilinear ×2: 64×64','Bilinear ×4: 128×128','PE: bilinear ×14, 448×448','PE: horizontal flip']
    for ax,im,title in zip(axes[row],views,titles):
        ax.imshow(im,interpolation='nearest'); ax.set_title(title,fontsize=10)
    axes[row,0].set_title(f"ID {int(data['test_ids'][i])} | 32×32",fontsize=10)
fig.suptitle('Same source image at different resolutions',fontsize=14)
plt.tight_layout(); plt.show()

## 8. Full image and horizontal flip

For each image, I average PE's ten class scores from the full image and its horizontal flip. The reported CSV uses these two views.

The official normalization is $x_{norm}=(x/255-0.5)/0.5$ per channel. The next cell compares the illustrated transform with the official implementation without loading the weights.


In [ ]:
pe.prepare_pe_source(Path('/content/recvis_pe_source'))
from core.vision_encoder.transforms import get_image_transform
reference_transform = get_image_transform(448,center_crop=False)
example = Image.fromarray(data['test'][int(test_idx[0])])
x = reference_transform(example)
rgb = np.asarray(bilinear_rgb(np.asarray(example),448),dtype=np.float32)/255
manual = (torch.from_numpy(rgb.copy()).permute(2,0,1)-0.5)/0.5
assert torch.allclose(x,manual,atol=1e-6), 'Manual preprocessing differs from the official transform.'
print('Official tensor:',tuple(x.shape),'| range:',float(x.min()),float(x.max()))
print('Maximum difference from official transform:',float((x-manual).abs().max()))

## 9. Text prototypes and class scores

For each class $c$, I insert its name into the 80 public OpenCLIP templates. Each text embedding is normalized, then the average is normalized again:

$$t_c=\operatorname{norm}\left(\frac{1}{80}\sum_{k=1}^{80}\operatorname{norm}(g(q_k(c)))\right).$$

For each view, $z_v=\operatorname{norm}(f(v(x)))$. The class score is the mean cosine similarity across the two views:

$$s_c(x)=\frac{\langle z_{original},t_c\rangle+\langle z_{flip},t_c\rangle}{2},\qquad \hat y(x)=\arg\max_c s_c(x).$$

There is no trained CIFAR classification head. Zero-shot here refers to using the pretrained encoders directly for these classes; PE itself was pretrained with image-text supervision.


In [ ]:
PROMPT_DIR = Path('/content/recvis_pe_didactic_prompts')
templates,prompt_provenance = sv.load_prompt_bank(PROMPT_DIR)
print(len(templates),'plantillas ×',len(CLASSES),'clases =',len(templates)*len(CLASSES),'textos')
display(pd.DataFrame({'Public templates (first eight)':templates[:8]}))
print('Class order:',list(enumerate(CLASSES)))

## 10. Calibration and evaluation

| Data | Use in this run |
|---|---|
| Training images | Throughput benchmark and batch sizing |
| Training labels | Example plots only |
| 20% of validation | Fit a positive temperature $T$ |
| 60% of validation | Accuracy, loss, confusion and error analysis |
| Remaining 20% | Retained split from the earlier experiment; unused for PE fitting |
| Test images and IDs | Final inference and CSV export |

$$p_c(x)=\operatorname{softmax}(s(x)/T)_c.$$

Temperature calibration changes confidence but preserves the argmax, so it does not change the labels in the PE CSV. The validation split is internal to this project; it is not a new external test set.


## 11. Run inference

Set **`RUN_INFERENCE=True`** to run full-image/flip PE inference. The PE/SigLIP blend, local crops and corruption diagnostics are disabled for this run.

The encoder is frozen, with FP32 weights, BF16 autocast and FP16 feature caches. The official image transform runs on CPU. I reuse the saved weight revision when available; otherwise the run records the revision it loads. The seven-hour budget starts with inference.


In [ ]:
if RUN_INFERENCE:
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('Select an A100 or another BF16-capable GPU for inference.')
    torch.manual_seed(42)
    torch.set_float32_matmul_precision('high')
    SETTINGS = dict(seed=42,previous_run=str(PREVIOUS_DIR),crop_size=28,uncertain_fraction=.25,crop_weight=.25,
                    run_local_views=False,run_stress_audit=False)
    # Crop settings are inactive but retained for module compatibility.
    class PEZeroShotOnly(pe.PEExperiment):
        def blend_previous(self):
            print('PE-only inference; SigLIP blending is disabled.')
    model_record = Path('models')/(pe.MODEL_ID.replace('/','__')+'.json')
    previous_record = PREVIOUS_DIR/model_record
    new_record = NEW_RUN_DIR/model_record
    if previous_record.exists():
        record=json.loads(previous_record.read_text())
        if record['model_id']!=pe.MODEL_ID:
            raise ValueError('The saved model record does not match PE.')
        if new_record.exists() and json.loads(new_record.read_text())!=record:
            raise ValueError('Weight revision mismatch; use a new run directory.')
        sv.atomic_json(record,new_record)
    budget=sv.Budget(7.0)
    experiment=PEZeroShotOnly(data,SETTINGS,NEW_RUN_DIR,BASE/'cache_pe_semantic',
                              Path('/content/recvis_pe_chunks'),budget)
    sv.atomic_json(dict(sources=SOURCES,pe_upstream_revision=pe.PE_REVISION),NEW_RUN_DIR/'code_revision.json')
    experiment.run()
    RESULT_DIR=NEW_RUN_DIR
else:
    RESULT_DIR=PREVIOUS_DIR
    print('Reading saved results; inference is disabled.')
print('Results:',RESULT_DIR)

## 12. Load saved probabilities

The following analysis uses `pe_zero_shot_probabilities.npz`. The loader checks data hashes, class order, splits and IDs before pairing predictions with images.

If the file is missing, update `PREVIOUS_PE_RUN` or enable inference above.


In [ ]:
P_VAL=P_TEST=None
PROB_FILE=RESULT_DIR/'pe_zero_shot_probabilities.npz'
METRIC_FILE=RESULT_DIR/'pe_zero_shot_metrics.json'
if PROB_FILE.exists():
    manifest=json.loads((RESULT_DIR/'experiment.json').read_text())
    assert manifest['data_hashes']==data['hashes'], 'Prediction data hashes do not match.'
    assert manifest['classes']==list(CLASSES), 'Class order mismatch.'
    assert manifest['validation_parts']==[p.tolist() for p in data['parts']], 'Split mismatch.'
    with np.load(PROB_FILE,allow_pickle=False) as saved:
        assert np.array_equal(saved['test_ids'],data['test_ids']), 'ID alignment mismatch.'
        val_p,test_p=saved['val'].copy(),saved['test'].copy()
    for split,prob in [('val',val_p),('test',test_p)]:
        assert prob.shape==(len(data[split]),10)
        assert np.isfinite(prob).all() and (prob>=0).all()
        assert np.allclose(prob.sum(axis=1),1,atol=2e-5)
    P_VAL,P_TEST=val_p,test_p
    print('Checked probabilities:',P_VAL.shape,P_TEST.shape)
    if METRIC_FILE.exists():
        metrics=json.loads(METRIC_FILE.read_text())
        print('Saved temperature:',metrics.get('temperature'))
else:
    print('File not found:',PROB_FILE)
    print('Update PREVIOUS_PE_RUN or enable RUN_INFERENCE and run its cell.')

## 13. Test predictions

Show the top three predictions for the same random images sampled earlier. The bars are calibrated model probabilities; the test labels are unavailable.


In [ ]:
if P_TEST is not None:
    fig,axes=plt.subplots(4,2,figsize=(11,11),gridspec_kw={'width_ratios':[1,2]},squeeze=False)
    for row,i in enumerate(test_idx[:4]):
        axes[row,0].imshow(bilinear_rgb(data['test'][i],448),interpolation='nearest')
        axes[row,0].axis('off');axes[row,0].set_title(f"ID {int(data['test_ids'][i])} — label unavailable")
        best=np.argsort(P_TEST[i])[-3:][::-1]
        axes[row,1].barh(np.arange(3),P_TEST[i,best],color=['#2563eb','#94a3b8','#cbd5e1'])
        axes[row,1].set_yticks(np.arange(3),[CLASSES[j] for j in best]);axes[row,1].invert_yaxis()
        axes[row,1].set_xlim(0,1);axes[row,1].set_xlabel('Calibrated probability')
    plt.tight_layout();plt.show()
else:
    print('Load the checked probabilities before running this section.')

## 14. Validation errors

Use the labeled evaluation split to plot a confusion matrix and a random sample of errors. This helps me inspect failure cases on CIFAR validation, while Kaggle measures performance on the synthetic domain.


In [ ]:
if P_VAL is not None:
    audit=data['parts'][2]
    truth=data['val_labels'][audit];pred=P_VAL[audit].argmax(axis=1)
    accuracy=float(np.mean(pred==truth))
    print(f'Validation evaluation accuracy: {accuracy:.3%} ({len(audit):,} images)')
    print('Recorded public Kaggle accuracy: 96.776%.')
    confusion=np.zeros((10,10),dtype=np.int64)
    np.add.at(confusion,(truth,pred),1)
    normalized=confusion/np.maximum(confusion.sum(axis=1,keepdims=True),1)
    fig,ax=plt.subplots(figsize=(9,7))
    im=ax.imshow(normalized,vmin=0,vmax=1,cmap='Blues')
    ax.set_xticks(range(10),CLASSES,rotation=45,ha='right');ax.set_yticks(range(10),CLASSES)
    ax.set_xlabel('Prediction');ax.set_ylabel('Validation label')
    ax.set_title('Confusion matrix normalized by true class')
    for r in range(10):
        for c in range(10):
            ax.text(c,r,f'{normalized[r,c]:.0%}',ha='center',va='center',fontsize=8,
                    color='white' if normalized[r,c]>.5 else 'black')
    fig.colorbar(im,ax=ax);plt.tight_layout();plt.show()
    errors=audit[pred!=truth]
    if len(errors):
        chosen=np.random.default_rng(SAMPLE_SEED).choice(errors,size=min(8,len(errors)),replace=False)
        fig,axes=plt.subplots(2,4,figsize=(12,6),squeeze=False)
        for ax in axes.flat:ax.axis('off')
        for ax,i in zip(axes.flat,chosen):
            ax.imshow(data['val'][i],interpolation='nearest')
            ax.set_title(f"True: {CLASSES[int(data['val_labels'][i])]}\nPE: {CLASSES[int(P_VAL[i].argmax())]}",fontsize=10)
        fig.suptitle('Random validation errors')
        plt.tight_layout();plt.show()
else:
    print('Load the checked probabilities before running this section.')

## 15. Result and next step

PE reached **96.776%** public accuracy, my best submission. I did not run separate Kaggle ablations for resolution or the horizontal flip, so their individual contributions remain unclear.

At this point I also wanted to check the cost of using such a large model. The next notebook switches to MobileCLIP2-S0, with about 11.4M visual/classifier parameters, to measure the accuracy loss with a much smaller encoder.


## 16. Export the PE submission

Regenerate the CSV from the checked probabilities, aligned to the IDs in `sample_submission.csv`, then download it for Kaggle.


In [ ]:
if P_TEST is not None:
    output=Path('/content/submission_pe_zero_shot.csv')
    sv.write_submission(data['test_ids'],P_TEST,data['files']['sample_submission.csv'],output)
    from google.colab import files
    files.download(str(output))
else:
    print('Load or compute the PE probabilities first.')

## References

- [Perception Encoder paper](https://arxiv.org/abs/2504.13181) and [PE-Core-G14-448 checkpoint](https://huggingface.co/facebook/PE-Core-G14-448).
- [Pinned architecture configuration](https://github.com/facebookresearch/perception_models/blob/3e352cca660658d4b5c90f42a7808b11469e4c66/core/vision_encoder/config.py) and [image transform](https://github.com/facebookresearch/perception_models/blob/3e352cca660658d4b5c90f42a7808b11469e4c66/core/vision_encoder/transforms.py).
- [OpenCLIP text templates](https://github.com/mlfoundations/open_clip/blob/4cc208d3a44a2fb6d6435006a46f520d2ccc29b1/src/open_clip/zero_shot_metadata.py), with the license included in the repository.
- [PE experiment module](https://github.com/rivas-vega/A3_Recv/blob/c30fc5422f23728a42cdc18741dac7ef52a7154c/pe_semantic.py).

Each run saves data hashes, code and model revisions, temperature, probabilities and IDs alongside its results.
